# 🌳 4주차 실습 | AI 기초 강의
## 머신러닝 모델 직접 만들기 — Decision Tree

---

### 📌 오늘 실습 흐름 (슬라이드와 1:1 매칭)

| STEP | 내용 | 슬라이드 |
|------|------|----------|
| STEP 0 | 준비 — 라이브러리 + 한글 폰트 | S0 |
| STEP 1 | 데이터 불러오기 | S1 |
| STEP 2 | 결측값 확인 및 처리 | S2 |
| STEP 3 | 범주형 → 숫자 변환 (인코딩) | S3 |
| STEP 4 | Feature / Label 분리 | S4 |
| STEP 5 | 훈련 / 테스트 데이터 분리 | S5 |
| STEP 6 | 모델 학습 (fit) | S6 |
| STEP 7 | 예측 (predict) | S7 |
| STEP 8 | 성능 평가 | S8 |
| STEP 9 | 트리 시각화 | S9 |
| STEP 10 | depth 바꿔보기 — 과적합 확인 | S10 |

---

> **실행 방법**: 셀 왼쪽 ▶ 버튼 클릭 또는 `Shift + Enter`  
> ⚠️ **반드시 위에서 아래 순서대로 실행하세요!**

---
## 📦 STEP 0. 준비 — 가장 먼저 실행하세요

> ⚠️ 이 셀을 먼저 실행해야 아래 셀이 정상 작동합니다.  
> 한글 폰트 설치가 포함되어 있어 **30초 정도** 걸릴 수 있어요.

In [ ]:
# ── 라이브러리 불러오기 ───────────────────────────────────────────
import subprocess, glob, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
warnings.filterwarnings('ignore')

# ── 한글 폰트 설치 및 적용 ────────────────────────────────────────
subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-nanum'],
               capture_output=True)

nanum = [f for f in
         glob.glob('/usr/share/fonts/**/*.ttf', recursive=True)
         if 'NanumGothic.ttf' in f]

if nanum:
    fm.fontManager.addfont(nanum[0])
    prop = fm.FontProperties(fname=nanum[0])
    plt.rcParams['font.family'] = prop.get_name()
    print(f'✅ 한글 폰트 적용: {prop.get_name()}')
else:
    print('⚠️  나눔 폰트를 찾지 못했습니다.')

plt.rcParams['axes.unicode_minus'] = False
print('✅ 라이브러리 준비 완료! 이제 아래 셀을 순서대로 실행하세요.')

---
## 📂 STEP 1. 데이터 불러오기

연속형과 범주형 컬럼을 직접 확인합니다.

| 컬럼 | 타입 | 비고 |
|------|------|------|
| 공부시간 | **연속형** | 결측값 포함 |
| 출석률 | **연속형** | 결측값 포함 |
| 과제제출 | **연속형** | 정상 |
| 수면시간 | **연속형** | 정상 |
| 전공계열 | **범주형** | 이과/문과/예체능 → 인코딩 필요 |
| 합격여부 | **Label** | 0=불합격, 1=합격 |

In [ ]:
# 데이터 불러오기
df = pd.read_csv('student_data.csv')

print(f'📋 데이터 크기: {df.shape[0]}행 × {df.shape[1]}열')
print()
print('── 컬럼 타입 ──────────────────────────────')
print(df.dtypes)
print()
print('── 처음 5행 ───────────────────────────────')
df.head()

In [ ]:
# 기초 통계 확인 — 연속형 Feature의 범위와 분포를 봅니다
print('── 기초 통계 ──────────────────────────────')
df.describe().round(2)

---
## 🔍 STEP 2. 결측값 확인 및 처리

빠진 데이터를 채웁니다. **연속형 결측값 → 평균(mean)으로 채우기**

> 💡 결측값이 있는 채로 학습하면 오류가 납니다!

In [ ]:
# 결측값 확인
print('── 처리 전 결측값 ──────────────────────────')
print(df.isnull().sum())
print()

# 이상치 확인 (공부시간이 24시간 초과면 이상치)
outliers = df[df['공부시간'] > 24]
if len(outliers) > 0:
    print(f'⚠️  이상치 발견: 공부시간 > 24시간인 데이터 {len(outliers)}개')
    print(outliers[['공부시간']].to_string())
    print()

In [ ]:
# 이상치 처리: 24시간 초과 → NaN 처리 후 평균으로 대체
df.loc[df['공부시간'] > 24, '공부시간'] = np.nan

# 연속형 결측값 → 평균으로 채우기
for col in ['공부시간', '출석률']:
    mean_val = df[col].mean()
    df[col] = df[col].fillna(mean_val)
    print(f'{col} 결측값 → 평균 {mean_val:.2f}로 채움')

print()
print('── 처리 후 결측값 ──────────────────────────')
print(df.isnull().sum())
print()
print('✅ 결측값 처리 완료!')

---
## 🔄 STEP 3. 범주형 데이터 인코딩

**전공계열** (이과/문과/예체능) → 숫자로 변환합니다.

| 원래 값 | 변환 후 |
|---------|--------|
| 이과 | 0 |
| 문과 | 1 |
| 예체능 | 2 |

> ⚠️ 범주형을 그냥 넣으면 오류가 납니다. 인코딩은 필수예요!

In [ ]:
# 인코딩 전 확인
print('── 인코딩 전 ───────────────────────────────')
print(df['전공계열'].value_counts())
print()

# 범주형 → 숫자 변환
mapping = {'이과': 0, '문과': 1, '예체능': 2}
df['전공계열'] = df['전공계열'].map(mapping)

print('── 인코딩 후 ───────────────────────────────')
print(df['전공계열'].value_counts().sort_index())
print()
print('✅ 인코딩 완료! 이제 모든 컬럼이 숫자입니다.')
print()
df.head()

---
## ✂️ STEP 4. Feature / Label 분리

- **X (Feature)**: AI가 보고 판단하는 입력 정보
- **y (Label)**: AI가 맞혀야 할 정답

연속형 4개 + 범주형(인코딩됨) 1개 = 총 5개 Feature

In [ ]:
# Feature와 Label 분리
X = df[['공부시간', '출석률', '과제제출', '수면시간', '전공계열']]
y = df['합격여부']

print(f'X (Feature) shape: {X.shape}  ← {X.shape[0]}명 × {X.shape[1]}개 Feature')
print(f'y (Label)   shape: {y.shape}  ← {y.shape[0]}명의 정답')
print()
print(f'합격(1): {y.sum()}명  불합격(0): {(y==0).sum()}명')
print()
print('── Feature 미리보기 ───────────────────────')
X.head()

---
## ✂️ STEP 5. 훈련 / 테스트 데이터 분리

**처음 보는 데이터에서의 성능이 진짜 실력!**

- 훈련 데이터 80% → 모델이 공부하는 데이터
- 테스트 데이터 20% → 성능 평가용 (학습에 사용 ❌)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,     # 20%를 테스트로
    random_state=42    # 재현 가능하도록 고정
)

print(f'훈련 데이터: {len(X_train)}명 (80%)')
print(f'테스트 데이터: {len(X_test)}명 (20%)')
print()
print(f'훈련 합격 비율: {y_train.mean():.1%}')
print(f'테스트 합격 비율: {y_test.mean():.1%}')
print()
print('✅ 분리 완료!')

---
## 🌳 STEP 6. 모델 학습 (fit)

AI가 훈련 데이터를 보면서 **스스로 규칙을 만드는** 단계입니다.

> `max_depth=3`: 트리 깊이를 3으로 제한 → 과적합 방지

In [ ]:
# Decision Tree 모델 생성 및 학습
model = DecisionTreeClassifier(
    max_depth=3,      # 트리 깊이 제한
    random_state=42   # 재현 가능하도록 고정
)

model.fit(X_train, y_train)

print('✅ 학습 완료!')
print(f'   사용한 Feature: {list(X.columns)}')
print(f'   트리 깊이(max_depth): {model.get_depth()}')
print(f'   잎 노드 수: {model.get_n_leaves()}')

---
## 🔮 STEP 7. 예측 (predict)

학습된 모델이 **처음 보는 테스트 데이터**를 보고 합격/불합격을 판단합니다.

In [ ]:
# 예측
y_pred = model.predict(X_test)

print(f'실제값: {list(y_test[:10])}')
print(f'예측값: {list(y_pred[:10])}')
print()

# 맞춘 것 / 틀린 것 시각화
correct = (y_test == y_pred).sum()
wrong   = (y_test != y_pred).sum()
print(f'맞게 예측: {correct}개  |  틀리게 예측: {wrong}개')

# 결과 미리보기
result_df = X_test.copy()
result_df['실제값'] = y_test.values
result_df['예측값'] = y_pred
result_df['맞음여부'] = ['✅' if a==p else '❌'
                         for a, p in zip(y_test, y_pred)]
result_df.head(10)

---
## 📊 STEP 8. 성능 평가

3주차에서 배운 지표를 **그대로** 사용합니다.

| 지표 | 공식 | 의미 |
|------|------|------|
| Accuracy | (TP+TN) ÷ 전체 | 전체 중 맞춘 비율 |
| Precision | TP ÷ (TP+FP) | 합격 예측 중 진짜 합격 |
| Recall | TP ÷ (TP+FN) | 실제 합격자 중 놓치지 않은 비율 |
| F1 Score | 2×P×R ÷ (P+R) | Precision·Recall 균형 |

In [ ]:
# 성능 지표 계산
acc  = accuracy_score( y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(   y_test, y_pred, zero_division=0)
f1   = f1_score(       y_test, y_pred, zero_division=0)

print('=' * 42)
print(f'  Accuracy  (정확도) : {acc:.2f}  ({acc*100:.1f}%)')
print(f'  Precision (정밀도) : {prec:.2f}  ({prec*100:.1f}%)')
print(f'  Recall    (재현율) : {rec:.2f}  ({rec*100:.1f}%)')
print(f'  F1 Score  (균형)   : {f1:.2f}  ({f1*100:.1f}%)')
print('=' * 42)

# 혼동행렬
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
print(f'\n혼동행렬:')
print(f'  [[TN={cm[0,0]}, FP={cm[0,1]}],   ← 실제 불합격')
print(f'   [FN={cm[1,0]}, TP={cm[1,1]}]]   ← 실제 합격')

In [ ]:
# 지표 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 막대 차트
labels_bar = ['Accuracy\n(정확도)', 'Precision\n(정밀도)',
              'Recall\n(재현율)', 'F1 Score']
values_bar = [acc, prec, rec, f1]
colors_bar = ['#4A90D9', '#5CB8B2', '#F07C3A', '#4CAF82']

bars = axes[0].bar(labels_bar, values_bar,
                   color=colors_bar, edgecolor='white', width=0.5)
for bar, v in zip(bars, values_bar):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 v + 0.02, f'{v:.2f}',
                 ha='center', fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1.15)
axes[0].set_title('평가 지표', fontsize=14, fontweight='bold')
axes[0].axhline(y=0.8, color='gray', linestyle='--', alpha=0.4)
axes[0].grid(True, alpha=0.3, axis='y')

# 혼동행렬 시각화
cm_colors = [['#5CB8B2', '#E05C5C'], ['#F9C642', '#4CAF82']]
cm_labels = [[f'TN\n{cm[0,0]}', f'FP\n{cm[0,1]}'],
             [f'FN\n{cm[1,0]}', f'TP\n{cm[1,1]}']]
for i in range(2):
    for j in range(2):
        axes[1].add_patch(plt.Rectangle(
            (j, 1-i), 1, 1,
            facecolor=cm_colors[i][j], alpha=0.85,
            edgecolor='white', linewidth=3))
        axes[1].text(j+0.5, 1.5-i, cm_labels[i][j],
                     ha='center', va='center',
                     fontsize=18, fontweight='bold', color='white')
axes[1].set_xlim(0, 2)
axes[1].set_ylim(0, 2)
axes[1].set_xticks([0.5, 1.5])
axes[1].set_xticklabels(['합격(1) 예측', '불합격(0) 예측'])
axes[1].set_yticks([0.5, 1.5])
axes[1].set_yticklabels(['실제 불합격(0)', '실제 합격(1)'])
axes[1].set_title('혼동행렬', fontsize=14, fontweight='bold')
axes[1].xaxis.tick_top()

plt.tight_layout()
plt.show()

---
## 🌳 STEP 9. 트리 시각화

AI가 스스로 만들어낸 판단 규칙을 **눈으로** 확인합니다.

> 💡 **루트 노드(최상단)** 에 어떤 Feature가 있는지 확인해보세요.  
> AI가 가장 중요하게 본 기준입니다!

In [ ]:
# 트리 시각화
fig, ax = plt.subplots(figsize=(18, 7))

plot_tree(
    model,
    feature_names=X.columns,
    class_names=['불합격', '합격'],
    filled=True,          # 클래스별 색상 채우기
    rounded=True,         # 둥근 모서리
    fontsize=11,
    ax=ax
)

ax.set_title(
    f'Decision Tree 시각화  (max_depth={model.get_depth()})',
    fontsize=15, fontweight='bold', pad=20
)
plt.tight_layout()
plt.show()

print()
print('🔍 확인해보세요:')
print('  - 루트 노드(맨 위)의 Feature는 무엇인가요?')
print('  - 연속형 Feature는 어떤 숫자를 기준으로 나뉘나요?')
print('  - 파란색 노드 = 합격 쪽, 주황색 노드 = 불합격 쪽')

In [ ]:
# Feature Importance 시각화
importances = model.feature_importances_
feat_names  = list(X.columns)

# 중요도 높은 순서로 정렬
sorted_idx  = np.argsort(importances)[::-1]

print('── Feature Importance ─────────────────────')
for i in sorted_idx:
    bar = '█' * int(importances[i] * 40)
    print(f'  {feat_names[i]:<8}: {importances[i]:.3f}  {bar}')
print()
print(f'  합계: {importances.sum():.3f}  (항상 1.0)')

# 막대 차트
fig, ax = plt.subplots(figsize=(8, 4))
colors_fi = ['#4A90D9','#5CB8B2','#F07C3A','#7D3C98','#6B8CAE']
bars = ax.barh(
    [feat_names[i] for i in sorted_idx],
    [importances[i] for i in sorted_idx],
    color=[colors_fi[i] for i in sorted_idx],
    edgecolor='white'
)
for bar, v in zip(bars, [importances[i] for i in sorted_idx]):
    ax.text(v + 0.005, bar.get_y() + bar.get_height()/2,
            f'{v:.3f}', va='center', fontsize=12, fontweight='bold')
ax.set_xlim(0, max(importances) * 1.3)
ax.set_title('Feature Importance\n(어떤 Feature가 가장 중요했나?)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('중요도')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## 🎮 STEP 10. depth 바꿔보기 — 과적합 직접 확인

아래 `depth` 값을 바꿔가며 실행해보세요!

| depth 값 | 예상 결과 |
|---------|----------|
| `1` | 너무 단순 — 훈련·테스트 모두 낮음 |
| `3` | 적절 — 균형 ✅ |
| `5` | 과적합 시작 |
| `10` | 심각한 과적합 — 훈련 ↑↑ 테스트 ↓↓ |
| `None` | 제한 없음 — 최대 과적합 |

In [ ]:
# ✏️ 이 숫자를 바꿔보세요!
depth = 3

# ── 아래는 수정하지 마세요 ─────────────────────────────────────
m = DecisionTreeClassifier(max_depth=depth, random_state=42)
m.fit(X_train, y_train)

train_acc = m.score(X_train, y_train)
test_acc  = m.score(X_test,  y_test)
gap       = train_acc - test_acc

print(f'depth = {depth}')
print(f'  훈련 정확도: {train_acc:.3f}  ({train_acc*100:.1f}%)')
print(f'  테스트 정확도: {test_acc:.3f}  ({test_acc*100:.1f}%)')
print(f'  차이 (훈련-테스트): {gap:.3f}')

if gap > 0.1:
    print('  ⚠️  과적합 의심! 훈련·테스트 차이가 큽니다.')
elif gap < 0.02:
    print('  ✅  훈련·테스트가 균형 잡혀 있습니다.')
else:
    print('  🟡  약간의 차이가 있습니다.')

In [ ]:
# depth 1~15 범위에서 과적합 곡선 그리기
depths       = list(range(1, 16))
train_accs   = []
test_accs    = []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    train_accs.append(m.score(X_train, y_train))
    test_accs.append( m.score(X_test,  y_test))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depths, train_accs, 'o-', color='#4A90D9',
        linewidth=2.5, markersize=7, label='훈련 정확도')
ax.plot(depths, test_accs,  's-', color='#E05C5C',
        linewidth=2.5, markersize=7, label='테스트 정확도')

# 현재 선택한 depth 표시
if depth <= 15:
    ax.axvline(x=depth, color='gray', linestyle='--',
               alpha=0.6, label=f'현재 depth={depth}')

ax.set_xlabel('max_depth', fontsize=13)
ax.set_ylabel('정확도', fontsize=13)
ax.set_title('depth에 따른 훈련/테스트 정확도 변화\n(과적합 확인)',
             fontsize=14, fontweight='bold')
ax.set_ylim(0.5, 1.05)
ax.set_xticks(depths)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

# 과적합 구간 음영
best_test_depth = depths[test_accs.index(max(test_accs))]
ax.axvspan(best_test_depth + 0.5, 15.5,
           alpha=0.08, color='red', label='과적합 구간')
ax.annotate('과적합\n구간', xy=(13, 0.58),
            fontsize=11, color='#E05C5C', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\n📌 테스트 정확도가 가장 높은 depth: {best_test_depth}')
print(f'   이 depth에서 테스트 정확도: {max(test_accs):.3f}')

---
## ✅ 실습 완료!

### 오늘 배운 것 정리

| 단계 | 핵심 내용 |
|------|-----------|
| **전처리** | 결측값 → 평균으로, 범주형 → 인코딩으로 |
| **분리** | Feature(X) / Label(y), 훈련 80% / 테스트 20% |
| **학습** | `model.fit(X_train, y_train)` |
| **예측** | `model.predict(X_test)` |
| **평가** | Accuracy · Precision · Recall · F1 |
| **시각화** | `plot_tree()` — AI가 찾은 규칙 확인 |
| **과적합** | depth ↑ → 훈련 정확도 ↑↑, 테스트 정확도 ↓↓ |

---

### 🚀 다음 시간 예고
**Random Forest** — Decision Tree를 여러 개 만들어 투표로 결정하는 모델  
오늘 배운 파이프라인이 그대로 이어집니다!

---
*AI 기초 강의 | 4주차 실습 완료* 🎉